In [ ]:
# validate_medium.py
import json
import sys
from collections import defaultdict

path = "dataset/scienceworld/low_data/medium0_seed1.json"

def load_json(p):
    with open(p, 'r') as f:
        return json.load(f)

def print_overview(d):
    print(f"Top-level keys: {list(d.keys())}")
    lens = {k: (len(v) if isinstance(v, list) else None) for k, v in d.items()}
    print("Lengths per key:")
    for k, l in lens.items():
        print(f"  {k}: {l}")
    return lens

def check_elementwise_consistency(d):
    # If all values are lists, check each index i to see if inner shapes align
    keys = list(d.keys())
    list_keys = [k for k in keys if isinstance(d[k], list)]
    if not list_keys:
        print("No list-valued keys found.")
        return

    max_len = max(len(d[k]) for k in list_keys)
    min_len = min(len(d[k]) for k in list_keys)
    print(f"List lengths: min={min_len}, max={max_len}")

    # Report keys that differ from max_len
    mismatched_keys = [k for k in list_keys if len(d[k]) != max_len]
    if mismatched_keys:
        print("Keys with lengths != max_len:")
        for k in mismatched_keys:
            print(f"  {k}: {len(d[k])}")
    else:
        print("All top-level list keys have same length.")

    # Now check per-index: for each i in range(max_len) report missing / not-list issues
    problems = defaultdict(list)
    for i in range(max_len):
        for k in list_keys:
            lst = d[k]
            if i >= len(lst):
                problems[i].append(f"missing key '{k}' at index {i}")
            else:
                v = lst[i]
                # if we expect inner sequences (like action being a list of steps), optionally check type
                # we just print the type/length for inspection
                if isinstance(v, list):
                    pass
                # else keep as-is
    if problems:
        print("\nPer-index problems (some keys missing at some indices):")
        for i, issues in problems.items():
            print(f" index {i}: {issues[:5]}")
    else:
        print("No per-index missing-key problems detected (every index exists for each top-level list key).")

def sample_entries(d, n=3):
    print("\nSample entries (first n indices) per key:")
    keys = list(d.keys())
    for i in range(min(n, max((len(v) for v in d.values() if isinstance(v, list)), default=0))):
        print(f"\n--- index {i} ---")
        for k in keys:
            if isinstance(d[k], list) and i < len(d[k]):
                v = d[k][i]
                # print small summary
                if isinstance(v, list):
                    print(f"  {k}: list len={len(v)} sample={v[:2]}")
                else:
                    s = str(v)
                    print(f"  {k}: type={type(v).__name__} repr={s[:120] + ('...' if len(s) > 120 else '')}")
            else:
                print(f"  {k}: <missing at index {i}>")

def normalize_and_write(d, out_path):
    # Determine expected keys (use union of keys you want)
    expected_keys = ["subtask", "obs", "action", "reward", "score", "done"]
    # If file has other keys, include them
    expected_keys = list(dict.fromkeys(list(d.keys()) + expected_keys))

    existing_lengths = [len(v) for k, v in d.items() if isinstance(v, list)]
    max_len = max(existing_lengths) if existing_lengths else 0

    # Ensure keys exist
    for k in expected_keys:
        d.setdefault(k, [])

    # Normalize lengths
    for k in expected_keys:
        seq = d[k]
        if not isinstance(seq, list):
            seq = [seq]
        if len(seq) < max_len:
            # pick placeholder
            if k in ("subtask", "obs"):
                placeholder = ""
            elif k == "action":
                placeholder = []
            elif k in ("reward", "score"):
                placeholder = 0.0
            elif k == "done":
                placeholder = False
            else:
                placeholder = None
            d[k] = seq + [placeholder] * (max_len - len(seq))
        elif len(seq) > max_len:
            d[k] = seq[:max_len]

    with open(out_path, "w") as f:
        json.dump(d, f, indent=4)
    print(f"Normalized and wrote to {out_path}")

d = load_json(path)
lens = print_overview(d)
check_elementwise_consistency(d)
sample_entries(d, n=5)

# If you want to normalize and overwrite (or write to new file), uncomment below:
answer = input("\nDo you want to normalize & rewrite this JSON to a new file? [y/N]: ").strip().lower()
if answer == 'y':
    out = path.replace(".json", ".normalized.json")
    normalize_and_write(d, out)
    print("Done. Re-run the loader after inspecting the normalized file if needed.")


JSONDecodeError: Expecting property name enclosed in double quotes: line 3924 column 1 (char 450847)

In [ ]:
path = "dataset/scienceworld/low_data/expert.json"
df = pd.read_json(path)

rows_ci = df[df['task_description'].str.contains('Your task is to boil apple juice', case=False, na=False)]

print(rows_ci.columns)

In [8]:
import pandas as pd

df = pd.read_json("dataset/scienceworld/high_data/expert.json")
df = df.head(1)
# fully print every element of the first row of the df
for column in df.columns:
    print(df[column])


0    Task Description:\nYour task is to boil orange...
Name: task_description, dtype: object
0    [Navigate to kitchen, Prepare tools for measur...
Name: subtask, dtype: object
0    [This room is called the hallway. In it, you s...
Name: obs, dtype: object
0    [open door to kitchen, go to kitchen, look aro...
Name: action, dtype: object
0    [[open door to kitchen, go to kitchen], [look ...
Name: group_action, dtype: object
0    [The door is now open., You move to the kitche...
Name: next_obs, dtype: object
0    [0.0, 0.03, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.67...
Name: reward, dtype: object
0    [0.0, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03...
Name: score, dtype: object
0    [False, False, False, False, False, False, Fal...
Name: done, dtype: object


In [12]:
df = pd.read_json("dataset/scienceworld/high_data/medium0_seed1.json")
# fully print every element of the first row of the df
row = df.iloc[5].obs
print(row)

['Group action: []. Current observation: This room is called the hallway. In it, you see: \n\tthe agent\n\ta substance called air\n\ta drawing\nYou also see:\n\tA door to the art studio (that is closed)\n\tA door to the bedroom (that is closed)\n\tA door to the greenhouse (that is closed)\n\tA door to the kitchen (that is closed)\n\tA door to the living room (that is closed)\n\tA door to the workshop (that is closed)', "Group action: ['None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', '